### module 2

---



In [1]:
%pip install torch

In [2]:
# moving device to gpu (runtime should be set to t4 gpu)
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

Device: cuda


In [4]:
# uploading file(from file explorer) and creating documents

from google.colab import files
import pandas as pd

uploaded = files.upload()

file_name = next(iter(uploaded))

documents = pd.read_csv(file_name)

Saving all_papers_sections_clean.csv to all_papers_sections_clean.csv


In [5]:
# verifying interface
print("Shape:", documents.shape)
print("Columns:", documents.columns.tolist())
print()
print(documents.head())

Shape: (354, 4)
Columns: ['paper_id', 'heading_level', 'section_title', 'text']

  paper_id heading_level       section_title  \
0      P01            h2      1 Introduction   
1      P01            h2           2 Methods   
2      P01            h4  RAG-Sequence Model   
3      P01            h4     RAG-Token Model   
4      P01            h3  2.2 Retriever: DPR   

                                                text  
0  Pre-trained neural language models have been s...  
1  We explore RAG models, which use the input seq...  
2  The RAG-Sequence model uses the same retrieved...  
3  In the RAG-Token model we can draw a different...  
4  The retrieval component p η ( z | x ) p_{\eta}...  


In [7]:
# loading tokenizer and model

from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained(
    "BAAI/bge-small-en-v1.5"
)

model = AutoModel.from_pretrained(
    "BAAI/bge-small-en-v1.5"
)

model = model.to(device)
model.eval()

text = documents.iloc[0]["text"]

tokens = tokenizer.encode(
    text,
    add_special_tokens=False # no CLS , SEP tokens
)

print("Section:", documents.iloc[0]["section_title"])
print("Number of tokens:", len(tokens))
print("First 20 token IDs:", tokens[:20])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (825 > 512). Running this sequence through the model will result in indexing errors


Section: 1 Introduction
Number of tokens: 825
First 20 token IDs: [3653, 1011, 4738, 15756, 2653, 4275, 2031, 2042, 3491, 2000, 4553, 1037, 6937, 3815, 1997, 1999, 1011, 5995, 3716, 2013]


In [8]:
#  tokenization function
def tokenize_text(text):
    return tokenizer.encode(
        text,
        add_special_tokens=False
    )
# testing the function
tokens = tokenize_text(documents.iloc[0]["text"])

print("Number of tokens:", len(tokens))
print("First 20 tokens:", tokens[:20])


Number of tokens: 825
First 20 tokens: [3653, 1011, 4738, 15756, 2653, 4275, 2031, 2042, 3491, 2000, 4553, 1037, 6937, 3815, 1997, 1999, 1011, 5995, 3716, 2013]


In [9]:
# chunk function (with metadats of chunks)

def create_chunks(
    text,
    chunk_size=500,
    overlap=100,
    paper_id=None,
    section=None,
    start_chunk_number=1
):
    if overlap >= chunk_size:
        raise ValueError("Overlap must be smaller than chunk size.")

    tokens = tokenize_text(text)

    chunks = []

    start = 0
    chunk_number = start_chunk_number

    while start < len(tokens):
        end = start + chunk_size

        chunk_tokens = tokens[start:end]

        chunk_text = tokenizer.decode(
            chunk_tokens,
            skip_special_tokens=True
        )

        chunk = {
            "chunk_id": f"C{chunk_number:06d}",
            "paper_id": paper_id,
            "section": section,
            "text": chunk_text
        }

        chunks.append(chunk)

        start = end - overlap
        chunk_number += 1

    return chunks, chunk_number

# testing function

text = documents.iloc[0]["text"]

chunks, next_chunk_number = create_chunks(
    text,
    chunk_size=250,
    overlap=50,
    paper_id=documents.iloc[0]["paper_id"],
    section=documents.iloc[0]["section_title"],
    start_chunk_number=1
)

print("Number of chunks:", len(chunks))
print("Next chunk number:", next_chunk_number)

print("\nFirst chunk:")
print(chunks[0])

Number of chunks: 5
Next chunk number: 6

First chunk:
{'chunk_id': 'C000001', 'paper_id': 'P01', 'section': '1 Introduction', 'text': 'pre - trained neural language models have been shown to learn a substantial amount of in - depth knowledge from data. they can do so without any access to an external memory, as a parameterized implicit knowledge base. while this development is exciting, such models do have downsides : they cannot easily expand or revise their memory, can ’ t straightforwardly provide insight into their predictions, and may produce “ hallucinations ”. hybrid models that combine parametric memory with non - parametric ( i. e., retrieval - based ) memories can address some of these issues because knowledge can be directly revised and expanded, and accessed knowledge can be inspected and interpreted. realm and orqa, two recently introduced models that combine masked language models with a differentiable retriever, have shown promising results, but have only explored open 

In [10]:
# loop for all sections

all_chunks = []

next_chunk_number = 1

for _, row in documents.iterrows():

    chunks, next_chunk_number = create_chunks(
        text=row["text"],
        chunk_size=250,
        overlap=50,
        paper_id=row["paper_id"],
        section=row["section_title"],
        start_chunk_number=next_chunk_number
    )

    all_chunks.extend(chunks)

print("Total chunks:", len(all_chunks))
print("Next chunk number:", next_chunk_number)

print(all_chunks[0])
print()
print(all_chunks[1])
print()
print(all_chunks[-1])

# converting all_chunks list to dataframe

chunks_df = pd.DataFrame(all_chunks)

print("Shape:", chunks_df.shape)
print("Columns:", chunks_df.columns.tolist())

display(chunks_df.head())


Total chunks: 758
Next chunk number: 759
{'chunk_id': 'C000001', 'paper_id': 'P01', 'section': '1 Introduction', 'text': 'pre - trained neural language models have been shown to learn a substantial amount of in - depth knowledge from data. they can do so without any access to an external memory, as a parameterized implicit knowledge base. while this development is exciting, such models do have downsides : they cannot easily expand or revise their memory, can ’ t straightforwardly provide insight into their predictions, and may produce “ hallucinations ”. hybrid models that combine parametric memory with non - parametric ( i. e., retrieval - based ) memories can address some of these issues because knowledge can be directly revised and expanded, and accessed knowledge can be inspected and interpreted. realm and orqa, two recently introduced models that combine masked language models with a differentiable retriever, have shown promising results, but have only explored open - domain extra

,chunk_id,paper_id,section,text
0,C000001,P01,1 Introduction,pre - trained neural language models have been...
1,C000002,P01,1 Introduction,- sequence ( seq2seq ) models. we endow pre - ...
2,C000003,P01,1 Introduction,responsible for all tokens ) or a per - token ...
3,C000004,P01,1 Introduction,"the - art results on open natural questions, w..."
4,C000005,P01,1 Introduction,/ rag /. an interactive demo of rag models can...


In [11]:
# defining creat embedding window function

def create_embedding_windows(
    chunk,
    window_size=500
):

    tokens = tokenize_text(chunk["text"])

    windows = []

    start = 0
    window_number = 1

    while start < len(tokens):

        end = start + window_size

        window_tokens = tokens[start:end]

        window_text = tokenizer.decode(
            window_tokens,
            skip_special_tokens=True
        )

        window = {
            "window_id": f"{chunk['chunk_id']}_W{window_number:03d}",
            "parent_chunk_id": chunk["chunk_id"],
            "paper_id": chunk["paper_id"],
            "section": chunk["section"],
            "text": window_text,
            "token_ids": window_tokens
        }

        windows.append(window)

        start = end
        window_number += 1

    return windows

    # testing function for chunk size 250

test_chunk = chunks_df.iloc[0].to_dict()

test_windows = create_embedding_windows(test_chunk)

print("Parent chunk:", test_chunk["chunk_id"])
print("Number of windows:", len(test_windows))

for window in test_windows:
    print(window["window_id"])
    print("Parent:", window["parent_chunk_id"])
    print("Text tokens:", len(tokenize_text(window["text"])))
    print("Text:", window["text"][:200], "...")


Parent chunk: C000001
Number of windows: 1
C000001_W001
Parent: C000001
Text tokens: 250
Text: pre - trained neural language models have been shown to learn a substantial amount of in - depth knowledge from data. they can do so without any access to an external memory, as a parameterized implic ...


In [12]:
# testing the model on one window

test_window = test_windows[0]

print("Window:", test_window["window_id"])
print("Tokens:", len(test_window["token_ids"]))

import torch

inputs = tokenizer(
    test_window["text"],
    return_tensors="pt",
    truncation=True,
    max_length=512
)

inputs = {
    key: value.to(device)
    for key, value in inputs.items()
}
with torch.no_grad():
    outputs = model(**inputs)

print("Model output shape:", outputs.last_hidden_state.shape)


Window: C000001_W001
Tokens: 250
Model output shape: torch.Size([1, 252, 384])


In [13]:
# defining create embeddings function

import torch
import torch.nn.functional as F

def create_embedding(window):

# input and masking
    token_ids = window["token_ids"]

    input_ids = (
        [tokenizer.cls_token_id]
        + token_ids
        + [tokenizer.sep_token_id]
    )

    input_ids = torch.tensor(
        [token_ids],
        dtype=torch.long
    ).to(device)

    attention_mask = torch.ones_like(input_ids)

    # Get BGE's token representations

    with torch.no_grad():
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
    # CLS pooling

    embedding = outputs.last_hidden_state[:, 0]

    # Normalising

    embedding = F.normalize(
        embedding,
        p=2,
        dim=1
    )
    # Convert tensor to NumPy
    return embedding[0].cpu().numpy()

    # testing on one window

test_embedding = create_embedding(test_windows[0])

print("Embedding shape:", test_embedding.shape)
print("First 10 values:", test_embedding[:10])

Embedding shape: (384,)
First 10 values: [-0.05263321  0.02438411 -0.04428762  0.03380069 -0.00842582  0.02242123
 -0.00660181  0.00800332  0.04298164 -0.02196325]


In [14]:
# making all windows

all_windows = []

for _, chunk in chunks_df.iterrows():

    chunk = chunk.to_dict()

    windows = create_embedding_windows(
        chunk,
        window_size=500
    )

    all_windows.extend(windows)

print("Total embedding windows:", len(all_windows))
print(all_windows[0])
print(all_windows[-1])


Total embedding windows: 758
{'window_id': 'C000001_W001', 'parent_chunk_id': 'C000001', 'paper_id': 'P01', 'section': '1 Introduction', 'text': 'pre - trained neural language models have been shown to learn a substantial amount of in - depth knowledge from data. they can do so without any access to an external memory, as a parameterized implicit knowledge base. while this development is exciting, such models do have downsides : they cannot easily expand or revise their memory, can ’ t straightforwardly provide insight into their predictions, and may produce “ hallucinations ”. hybrid models that combine parametric memory with non - parametric ( i. e., retrieval - based ) memories can address some of these issues because knowledge can be directly revised and expanded, and accessed knowledge can be inspected and interpreted. realm and orqa, two recently introduced models that combine masked language models with a differentiable retriever, have shown promising results, but have only expl

In [15]:
# embedding all the windows

all_embeddings = []

for window in all_windows:

    embedding = create_embedding(window)

    all_embeddings.append(embedding)

print("Number of embeddings:", len(all_embeddings))
print("Embedding dimension:", all_embeddings[0].shape[0])

Number of embeddings: 758
Embedding dimension: 384


In [16]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 84.0 MB/s eta 0:00:00


In [17]:
# creating FAISS index ( retrival index )

import faiss
import numpy as np

# converting into matix
embedding_matrix = np.array(
    all_embeddings,
    dtype="float32"
)
# making faiss index
index = faiss.IndexFlatIP(384)

# adding vectors into faiss index
index.add(embedding_matrix)

print("Number of vectors in FAISS:", index.ntotal)
print("Vector dimension:", index.d)

Number of vectors in FAISS: 758
Vector dimension: 384


In [18]:
# defining create query embedding function

def create_query_embedding(query):

    query_instruction = (
        "Represent this sentence for searching relevant passages: "
    )

    query_text = query_instruction + query

    input_ids = tokenizer.encode(
        query_text,
        add_special_tokens=True
    )

    input_ids = torch.tensor(
        [input_ids],
        dtype=torch.long
    ).to(device)

    attention_mask = torch.ones_like(input_ids)

    with torch.no_grad():
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

    embedding = outputs.last_hidden_state[:, 0]

    embedding = F.normalize(
        embedding,
        p=2,
        dim=1
    )

    return embedding.cpu().numpy().astype("float32")

    # testing query embedding

query = "What is retrieval augmented generation?"

query_instruction = "Represent this sentence for searching relevant passages: "

query_with_instruction = query_instruction + query

query_embedding = create_query_embedding(query_with_instruction)

print("Query embedding shape:", query_embedding.shape)

Query embedding shape: (1, 384)


In [19]:
# testing top k retrieval

k = 5

distances, indices = index.search(
    query_embedding,
    k
)

print("Distances:")
print(distances)

print("Indices:")
print(indices)

Distances:
[[0.80659664 0.7821673  0.77356076 0.7626413  0.76001436]]
Indices:
[[209 130   1 178 210]]


In [20]:
# retriving metadata

retrieved_windows = []

for i in indices[0]:

    window = all_windows[i]

    retrieved_windows.append(window)

for rank, window in enumerate(retrieved_windows, start=1):

    print(f"\nRank {rank}")
    print("Window ID:", window["window_id"])
    print("Parent chunk:", window["parent_chunk_id"])
    print("Paper:", window["paper_id"])
    print("Section:", window["section"])
    print("Text:", window["text"][:300])


Rank 1
Window ID: C000210_W001
Parent chunk: C000210
Paper: P04
Section: 1 Introduction
Text: augmented generation ( rag ) lee et al. ( 2019 ) ; lewis et al. ( 2020 ) ; guu et al. ( 2020 ). answering a question then essentially involves retrieving relevant passages from a corpus and feeding these passages, along with the original question, to the lm. while initial approaches relied on specia

Rank 2
Window ID: C000131_W001
Parent chunk: C000131
Paper: P03
Section: I Introduction
Text: in retrieval, including indexing, query and embedding optimization. section iv concentrates on post - retrieval process and llm fine - tuning in generation. section v analyzes the three augmentation processes. section vi focuses on rag ’ s downstream tasks and evaluation system. section vii mainly d

Rank 3
Window ID: C000002_W001
Parent chunk: C000002
Paper: P01
Section: 1 Introduction
Text: - sequence ( seq2seq ) models. we endow pre - trained, parametric - memory generation models with a non - paramet

In [21]:
# defining batch embedding function

def create_embeddings(windows, batch_size=32):

    all_embeddings = []

    for start in range(0, len(windows), batch_size):

        batch = windows[start:start + batch_size]

        batch_input_ids = []

        for window in batch:

            token_ids = window["token_ids"]

            input_ids = (
                [tokenizer.cls_token_id]
                + token_ids
                + [tokenizer.sep_token_id]
            )

            batch_input_ids.append(input_ids)

        # Find longest sequence in this batch
        max_length = max(
            len(ids) for ids in batch_input_ids
        )

        padded_input_ids = []
        attention_masks = []

        for ids in batch_input_ids:

            padding_length = max_length - len(ids)

            padded_ids = (
                ids +
                [tokenizer.pad_token_id] * padding_length
            )

            attention_mask = (
                [1] * len(ids) +
                [0] * padding_length
            )

            padded_input_ids.append(padded_ids)
            attention_masks.append(attention_mask)

        input_ids = torch.tensor(
            padded_input_ids,
            dtype=torch.long
        ).to(device)

        attention_mask = torch.tensor(
            attention_masks,
            dtype=torch.long
        ).to(device)

        with torch.no_grad():

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

        # CLS representation
        embeddings = outputs.last_hidden_state[:, 0]

        # L2 normalization
        embeddings = F.normalize(
            embeddings,
            p=2,
            dim=1
        )

        all_embeddings.extend(
            embeddings.cpu().numpy()
        )

    return np.array(
        all_embeddings,
        dtype="float32"
    )

In [22]:
#  testing the batch embeding function

all_embeddings = create_embeddings(
    all_windows,
    batch_size=32
)

print("Number of embeddings:", len(all_embeddings))
print("Embedding dimension:", all_embeddings.shape[1])

Number of embeddings: 758
Embedding dimension: 384


In [23]:
# defining build_faiss_index

def build_faiss_index(embeddings):

    embedding_dimension = embeddings.shape[1]

    index = faiss.IndexFlatIP(
        embedding_dimension
    )

    index.add(embeddings)

    return index

In [24]:
index = build_faiss_index(all_embeddings)

print("Number of vectors:", index.ntotal)
print("Vector dimension:", index.d)

Number of vectors: 758
Vector dimension: 384


In [25]:
# creating chunk dictionary
parent_chunks = {
    chunk["chunk_id"]: chunk
    for chunk in chunks
}

In [26]:
# defining retrival function

def retrieve(
    query,
    index,
    windows,
    parent_chunks,
    top_k=5
):
    # Create query embedding
    query_embedding = create_query_embedding(query)

    # Search ALL embedding windows
    distances, indices = index.search(
        query_embedding,
        index.ntotal
    )

    retrieved_chunks = []
    seen_chunks = set()

    # Go through windows in similarity order
    for distance, idx in zip(distances[0], indices[0]):

        window = windows[idx]

        parent_chunk_id = window["parent_chunk_id"]

        # Skip if this parent chunk was already retrieved
        if parent_chunk_id in seen_chunks:
            continue

        # Mark parent chunk as retrieved
        seen_chunks.add(parent_chunk_id)

        # Get the original full parent chunk
        parent_chunk = parent_chunks[parent_chunk_id]

        retrieved_chunks.append({
            "rank": len(retrieved_chunks) + 1,
            "window_id": window["window_id"],
            "parent_chunk_id": parent_chunk_id,
            "paper_id": parent_chunk["paper_id"],
            "section": parent_chunk["section"],
            "text": parent_chunk["text"],
            "similarity_score": float(distance)
        })

        # Stop after Top-K DISTINCT parent chunks
        if len(retrieved_chunks) >= top_k:
            break

    return retrieved_chunks

In [27]:
# defining build_retrieval_resources function

def build_retrieval_resources(
    documents,
    chunk_size,
    overlap
):

    # 1. Create experimental chunks
    all_chunks = []
    next_chunk_number = 1

    for _, row in documents.iterrows():

        chunks, next_chunk_number = create_chunks(
            text=row["text"],
            chunk_size=chunk_size,
            overlap=overlap,
            paper_id=row["paper_id"],
            section=row["section_title"],
            start_chunk_number=next_chunk_number
        )

        all_chunks.extend(chunks)


    # 2. Create parent-chunk lookup
    parent_chunks = {
        chunk["chunk_id"]: chunk
        for chunk in all_chunks
    }


    # 3. Create embedding windows
    all_windows = []

    for chunk in all_chunks:

        windows = create_embedding_windows(
            chunk,
            window_size=500
        )

        all_windows.extend(windows)


    # 4. Create embeddings
    all_embeddings = create_embeddings(
        all_windows,
        batch_size=32
    )


    # 5. Build FAISS index
    index = build_faiss_index(
        all_embeddings
    )


    return (
        index,
        all_windows,
        parent_chunks
    )

In [28]:
# defining run_retrival function

def run_retrieval(
    documents,
    query,
    chunk_size,
    overlap,
    top_k
):

    index, all_windows, parent_chunks = build_retrieval_resources(
        documents,
        chunk_size,
        overlap
    )

    retrieved_chunks = retrieve(
        query=query,
        index=index,
        windows=all_windows,
        parent_chunks=parent_chunks,
        top_k=top_k
    )

    return retrieved_chunks

In [29]:
from google.colab import drive
drive.mount("/content/drive")

import os

checkpoint_dir = "/content/drive/MyDrive/RAG project/rag_modules"
os.makedirs(checkpoint_dir, exist_ok=True)

checkpoint_file = os.path.join(
    checkpoint_dir,
    "module_2_retrieval_checkpoint.csv"
)

print("Checkpoint file:", checkpoint_file)

Mounted at /content/drive
Checkpoint file: /content/drive/MyDrive/RAG project/rag_modules/module_2_retrieval_checkpoint.csv


In [30]:
# defining run_experiment function

def run_experiment(
    documents,
    queries,
    chunk_sizes,
    overlaps,
    top_k_values,
    checkpoint_file
):
    # Load any results saved by an earlier run
    if os.path.exists(checkpoint_file):
        saved = pd.read_csv(checkpoint_file)
        completed = set(saved["config_id"].unique())
        print(f"Resuming: {len(completed)} configurations already saved.")
    else:
        saved = pd.DataFrame()
        completed = set()
        print("Starting a new experiment.")

    config_number = 1

    for chunk_size in chunk_sizes:
        for overlap in overlaps:

            # Determine which configuration IDs belong to this
            # chunk-size / overlap combination.
            config_ids = [
                f"CFG_{config_number + i:03d}"
                for i in range(len(top_k_values))
            ]

            # Avoid rebuilding embeddings if all Top-K configurations
            # for this combination are already complete.
            if all(cid in completed for cid in config_ids):
                print(
                    f"Skipping completed chunk_size={chunk_size}, "
                    f"overlap={overlap}"
                )
                config_number += len(top_k_values)
                continue

            print(
                f"Building resources: chunk_size={chunk_size}, "
                f"overlap={overlap}"
            )

            index, all_windows, parent_chunks = (
                build_retrieval_resources(
                    documents, chunk_size, overlap
                )
            )

            max_top_k = max(top_k_values)

            for top_k in top_k_values:
                config_id = f"CFG_{config_number:03d}"
                config_number += 1

                # Resume by skipping fully saved configurations
                if config_id in completed:
                    print(f"Skipping {config_id}")
                    continue

                print(f"Running {config_id}, Top-K={top_k}")
                config_rows = []

                for query_item in queries:
                    query_id = query_item["query_id"]
                    query = query_item["query"]

                    results = retrieve(
                        query=query,
                        index=index,
                        windows=all_windows,
                        parent_chunks=parent_chunks,
                        top_k=max_top_k
                    )

                    for result in results[:top_k]:
                        config_rows.append({
                            "config_id": config_id,
                            "query_id": query_id,
                            "query": query,
                            "chunk_size": chunk_size,
                            "overlap": overlap,
                            "top_k": top_k,
                            "rank": result["rank"],
                            "window_id": result["window_id"],
                            "parent_chunk_id": result["parent_chunk_id"],
                            "paper_id": result["paper_id"],
                            "section": result["section"],
                            "text": result["text"],
                            "similarity_score": result["similarity_score"]
                        })

                # Save immediately after this configuration finishes
                config_df = pd.DataFrame(config_rows)

                if not config_df.empty:
                    config_df.to_csv(
                        checkpoint_file,
                        mode="a",
                        header=not os.path.exists(checkpoint_file),
                        index=False
                    )

                completed.add(config_id)
                print(
                    f"Saved {config_id}. "
                    f"Completed: {len(completed)}/"
                    f"{len(chunk_sizes) * len(overlaps) * len(top_k_values)}"
                )

    # Load the full checkpoint as the final result
    if os.path.exists(checkpoint_file):
        return pd.read_csv(checkpoint_file)

    return pd.DataFrame()


In [31]:

# Load the finalized evaluation question set

from google.colab import files

uploaded = files.upload()

query_file = next(
    name for name in uploaded
    if name.lower().endswith(".csv")
)

questions_df = pd.read_csv(query_file)

# Basic validation
required_columns = [
    "query_id",
    "question",
    "category",
    "expected_answer",
    "answerability"
]

missing_columns = [
    col for col in required_columns
    if col not in questions_df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing columns in query CSV: {missing_columns}"
    )

if questions_df["query_id"].duplicated().any():
    raise ValueError("Duplicate query IDs found.")

if questions_df["question"].isna().any():
    raise ValueError("Some questions are missing.")

# Pass each question's ID and text to Module 2
queries = (
    questions_df[["query_id", "question"]]
    .rename(columns={"question": "query"})
    .to_dict("records")
)

print("Questions loaded:", len(queries))
display(questions_df.head())


Saving evaluation_questions - module1_query_set.csv to evaluation_questions - module1_query_set.csv
Questions loaded: 30


,query_id,question,category,expected_answer,supporting_paper_ids,supporting_sections,answerability,reason_unanswerable
0,Q001,What problem does Retrieval-Augmented Generati...,direct,RAG combines a parametric language model with ...,P01,Introduction; Methods,Yes,NaN
1,Q002,What are the two main components combined in t...,direct,A pre-trained sequence-to-sequence generator a...,P01,Methods,Yes,NaN
2,Q003,What task does Dense Passage Retrieval (DPR) t...,direct,Open-domain question answering by retrieving r...,P02,Introduction; Methods,Yes,NaN
3,Q004,What is the main purpose of RAGAS?,direct,To provide automated evaluation of retrieval-a...,P04,Introduction; Methods,Yes,NaN
4,Q005,What issue is investigated in 'Lost in the Mid...,direct,The paper investigates how language models use...,P05,Introduction; Results,Yes,NaN


In [32]:
# running the run_experiment function


retrieval_results = run_experiment(
    documents=documents,
    queries=queries,
    chunk_sizes=[250, 500, 750, 1000, 1500],
    overlaps=[0, 50, 100, 200],
    top_k_values=[1, 3, 5, 10],
    checkpoint_file=checkpoint_file
)

print("Result shape:", retrieval_results.shape)
print("Unique questions:", retrieval_results["query_id"].nunique())
print("Unique configurations:", retrieval_results["config_id"].nunique())

display(retrieval_results.head())


Starting a new experiment.
Building resources: chunk_size=250, overlap=0
Running CFG_001, Top-K=1
Saved CFG_001. Completed: 1/80
Running CFG_002, Top-K=3
Saved CFG_002. Completed: 2/80
Running CFG_003, Top-K=5
Saved CFG_003. Completed: 3/80
Running CFG_004, Top-K=10
Saved CFG_004. Completed: 4/80
Building resources: chunk_size=250, overlap=50
Running CFG_005, Top-K=1
Saved CFG_005. Completed: 5/80
Running CFG_006, Top-K=3
Saved CFG_006. Completed: 6/80
Running CFG_007, Top-K=5
Saved CFG_007. Completed: 7/80
Running CFG_008, Top-K=10
Saved CFG_008. Completed: 8/80
Building resources: chunk_size=250, overlap=100
Running CFG_009, Top-K=1
Saved CFG_009. Completed: 9/80
Running CFG_010, Top-K=3
Saved CFG_010. Completed: 10/80
Running CFG_011, Top-K=5
Saved CFG_011. Completed: 11/80
Running CFG_012, Top-K=10
Saved CFG_012. Completed: 12/80
Building resources: chunk_size=250, overlap=200
Running CFG_013, Top-K=1
Saved CFG_013. Completed: 13/80
Running CFG_014, Top-K=3
Saved CFG_014. Completed

,config_id,query_id,query,chunk_size,overlap,top_k,rank,window_id,parent_chunk_id,paper_id,section,text,similarity_score
0,CFG_001,Q001,What problem does Retrieval-Augmented Generati...,250,0,1,1,C000594_W001,C000594,P10,1 Introduction,retrieval - augmented generation ( rag ) syste...,0.879269
1,CFG_001,Q002,What are the two main components combined in t...,250,0,1,1,C000114_W001,C000114,P03,I Introduction,how these components intricately collaborate t...,0.751279
2,CFG_001,Q003,What task does Dense Passage Retrieval (DPR) t...,250,0,1,1,C000077_W001,C000077,P02,5 Experiments: Passage Retrieval,"in this section, we evaluate the retrieval per...",0.816723
3,CFG_001,Q004,What is the main purpose of RAGAS?,250,0,1,1,C000166_W001,C000166,P03,VI-A Downstream Task,the core task of rag remains question answerin...,0.675176
4,CFG_001,Q005,What issue is investigated in 'Lost in the Mid...,250,0,1,1,C000153_W001,C000153,P03,IV-A Context Curation,redundant information can interfere with the f...,0.803355


In [33]:
final_file = "/content/drive/MyDrive/RAG project/rag_modules/module_2_retrieval_results.csv"

retrieval_results.to_csv(final_file, index=False)

print("Final results saved:", final_file)
print("Rows:", len(retrieval_results))
print("Unique questions:", retrieval_results["query_id"].nunique())
print("Unique configurations:", retrieval_results["config_id"].nunique())

print("Configuration counts:")
print(
    retrieval_results.groupby("config_id")["query_id"]
    .nunique()
    .value_counts()
)

Final results saved: /content/drive/MyDrive/RAG project/rag_modules/module_2_retrieval_results.csv
Rows: 11400
Unique questions: 30
Unique configurations: 80
Configuration counts:
query_id
30    80
Name: count, dtype: int64


In [34]:


print("Final file exists:", os.path.exists(final_file))
print("Final file path:", final_file)

print("Rows:", len(retrieval_results))
print("Questions:", retrieval_results["query_id"].nunique())
print("Configurations:", retrieval_results["config_id"].nunique())

Final file exists: True
Final file path: /content/drive/MyDrive/RAG project/rag_modules/module_2_retrieval_results.csv
Rows: 11400
Questions: 30
Configurations: 80


In [35]:
# Check how many questions each configuration covers
questions_per_config = (
    retrieval_results.groupby("config_id")["query_id"].nunique()
)

print("Configurations with all 30 questions:",
      (questions_per_config == 30).sum())

# Check how many retrieved chunks each question/configuration has
counts = retrieval_results.groupby(
    ["config_id", "query_id"]
).size()

print("Question/configuration pairs:", len(counts))
print("Pairs with zero rows: not represented in CSV")
print("Minimum rows per pair:", counts.min())
print("Maximum rows per pair:", counts.max())

# Check whether a question/configuration has duplicate ranks
print(
    "Duplicate ranks:",
    retrieval_results.duplicated(
        ["config_id", "query_id", "rank"]
    ).sum()
)

Configurations with all 30 questions: 80
Question/configuration pairs: 2400
Pairs with zero rows: not represented in CSV
Minimum rows per pair: 1
Maximum rows per pair: 10
Duplicate ranks: 0
